# CAM:macro_microphysics on its own

`macro_microphysics` is CAM5's cloud macrophysics + microphysics stage, one
box of the PI-atm workflow.  The workflow is a list, so the cell below assigns
a list holding only that process, sets whatever inputs it likes in the live
StatePool, and runs one step.  The step runs only that process, and CAM writes
its usual history file at the end of it.  The cell's value is that file's path.

`nhtfrq=1` makes CAM write a history sample every step.  Change the input
lines to set different inputs; everything else stays the same.


In [ ]:
import freecam as fc

# CAM:macro_microphysics on its own.  The workflow is a list; leave just that
# process in it, so one step is one call, and CAM writes its history file at
# the end of the step.
with fc.Driver(case='PI-atm', nsteps=1, namelist={'nhtfrq': 1}) as driver:
    driver.initialize()
    workflow = driver.cam.workflow
    workflow[:] = [workflow['macro_microphysics']]

    # inputs: any field, any value.  CAM carries dry static energy next to
    # temperature and re-derives one from the other, so warm both.
    state = driver.cam.state
    state.T += 2.0
    state['phys_state.s'] += 2.0 * 1004.64
    state.q[:, :, 0, :] *= 1.05
    # workflow['macro_microphysics'].properties['cldfrc_rhminl'] = 0.9

    driver.run()

driver.cam.history.latest()


## Or run it in place, without a step

Calling the process's own `run()` executes just that Fortran routine on the
live state -- no timestep, no coupling, no history write -- and the result is
read straight back from the StatePool.  This is the quick-inspection path; the
cell above is the one that leaves a NetCDF file behind.  It uses the offline
`PI-atm-replay` case, which has no live coupler to open and close around the
call.


In [ ]:
import freecam as fc

# Run this one process by itself -- no timestep, no coupling -- and read the
# change straight from the live StatePool.  PI-atm-replay drives CAM from a
# captured boundary, so there is no online coupler to complete first.
with fc.Driver(case='PI-atm-replay', nsteps=1) as driver:
    driver.initialize()
    state = driver.cam.state

    state.T += 2.0
    state['phys_state.s'] += 2.0 * 1004.64
    state.q[:, :, 0, :] *= 1.05

    before = state.T.mean(rank='global')
    driver.cam.workflow['macro_microphysics'].run()
    after = state.T.mean(rank='global')

    print(f'global-mean T: {before:.4f} K -> {after:.4f} K')
